# 11_Retrieval.ipynb
=================


This notebook demonstrates the semantic retrieval pipeline using MiniLM embeddings and FAISS vector indexing. We construct a semantic index from our support ticket corpus, run similarity queries, and compute standard information retrieval metrics (Recall@K, MRR, MAP).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT).exists():
    REPO_ROOT = REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pathlib import Path

from src.data.dataset import load_and_preprocess_dataset
from src.models.transformer.retrieval import SemanticRetriever
from src.utils.constants import OUTPUT_DIR

[07/13/26 17:03:41] INFO     Loading faiss with AVX2 support.

                    INFO     Could not load library with AVX2 support due to:                                      
                             ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")

                    INFO     Loading faiss.

                    INFO     Successfully loaded faiss.

## 1. Load Dataset Corpus
We load our preprocessed customer support ticket splits.

In [2]:
splits = load_and_preprocess_dataset()
test_df = splits["test"]
corpus = test_df["text"].tolist()
print(f"Corpus size: {len(corpus)} tickets")

[07/13/26 17:03:49] INFO     All dataset splits cached locally under:                                              
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1. Loading...

                    INFO     Loading 'train' split from local cache:                                               
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1\train.parquet

                    INFO     Loading 'val' split from local cache:                                                 
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1\val.parquet

                    INFO     Loading 'test' split from local cache:                                                
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1\test.parquet

Corpus size: 1302 tickets


## 2. Build FAISS Index
We initialize our `SemanticRetriever` and construct the index using Cosine Similarity.

In [3]:
retriever = SemanticRetriever(metric="cosine")
retriever.build_index(corpus)

[07/13/26 17:03:53] INFO     Loading sentence embedding model: all-MiniLM-L6-v2

                    INFO     Load pretrained SentenceTransformer: all-MiniLM-L6-v2

[07/13/26 17:03:58] INFO     Encoding corpus of 1302 texts...

                    INFO     FAISS index constructed successfully.

## 3. Perform Similarity Search
We run a query to locate semantically similar tickets from the corpus.

In [4]:
query = "My account is locked due to wrong passcode attempt"
results = retriever.retrieve(query, top_k=3)

for res in results:
    print(f"Rank {res['rank']} (Score: {res['score']:.4f}) [Index: {res['index']}]: {res['text']}")

Rank 1 (Score: 0.7687) [Index: 187]: help me reset the passcode.
Rank 2 (Score: 0.7622) [Index: 722]: how can i reset my passcode?
Rank 3 (Score: 0.7622) [Index: 102]: how can i reset my passcode ?


## 4. Evaluate Retrieval Quality
We define queries and use matching labels to build ground-truth indices. We then evaluate Recall@K, MRR, and MAP.

In [5]:
# Select top 10 tickets as validation queries
val_queries = test_df["text"].iloc[:10].tolist()
val_labels = test_df["label"].iloc[:10].tolist()

# Ground truth: other documents sharing the same category/label
ground_truth_indices = []
for label in val_labels:
    matching_ids = test_df[test_df["label"] == label].index.tolist()
    ground_truth_indices.append(matching_ids)

# Run evaluation
metrics = retriever.evaluate_retrieval(val_queries, ground_truth_indices, k=5)
for name, value in metrics.items():
    print(f"{name}: {value:.4%}")

Recall@5: 25.4670%
MRR: 100.0000%
MAP: 25.4670%


## 5. Serialise Retrieval Database
We save the index and corpus metadata to the disk.

In [6]:
index_dir = OUTPUT_DIR / "retrieval_index"
retriever.save_index(index_dir)
print(f"Index saved to: {index_dir}")

[07/13/26 17:04:21] INFO     Saved FAISS index and metadata to                                                     
                             C:\Users\gunav\Downloads\SupportAI\outputs\retrieval_index

Index saved to: C:\Users\gunav\Downloads\SupportAI\outputs\retrieval_index


In [7]:
# Export Phase Manifest
from src.api.app import get_git_commit
from src.utils.artifacts import save_yaml

manifest = {
    "phase": "11_Retrieval",
    "retrieval_status": "FAISS index saved and verified",
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_11_retrieval.yaml")
print("YAML manifest saved successfully:")
print(manifest)


[07/13/26 17:04:26] INFO     Saving YAML artifact to:                                                              
                             C:\Users\gunav\Downloads\SupportAI\outputs\manifests\phase_11_retrieval.yaml

YAML manifest saved successfully:
{'phase': '11_Retrieval', 'retrieval_status': 'FAISS index saved and verified', 'git_commit': '301987dee601e05014b46846834438d7a49c23d6'}
